In [ ]:
library(tidyverse)
library(pheatmap)

In [3]:
phe <- readRDS('phe3.Rds')

In [4]:
Allbinfreq <- read.table('All.each.hubs',header=T)

In [5]:
head(Allbinfreq)

,chr,start,end,U3008,U3009,U3013,U3028,U3031,U3039,U3042,U3054,U3073,U3078,U3085,U3086,U3118,U3121
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,chr1,773513,783513,0,0,0,0,0,0,0,0,1,0,0,1,0,0
2,chr1,812734,822734,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,chr1,822078,832078,0,0,0,1,1,0,0,1,2,0,0,0,0,0
4,chr1,884727,894727,1,0,0,0,0,0,0,0,0,0,0,1,0,0
5,chr1,918817,928817,0,0,0,1,0,0,0,1,1,0,0,3,0,0
6,chr1,936867,946867,0,0,0,2,2,0,0,1,1,0,0,0,0,0


In [5]:
phe  %>% filter(footprint=='C2')  %>% rownames()  %>% intersect(colnames(Allbinfreq)) -> C2names
phe  %>% filter(footprint=='C3')  %>% rownames()  %>% intersect(colnames(Allbinfreq)) -> C3names

In [1]:
C2names

ERROR: Error in eval(expr, envir, enclos): object 'C2names' not found


In [2]:
C3names

ERROR: Error in eval(expr, envir, enclos): object 'C3names' not found


In [8]:
paste(Allbinfreq$chr,Allbinfreq$start,Allbinfreq$end,sep="_") -> names
rownames(Allbinfreq) <- names

In [9]:
head(Allbinfreq)

,chr,start,end,U3008,U3009,U3013,U3028,U3031,U3039,U3042,U3054,U3073,U3078,U3085,U3086,U3118,U3121
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr1_773513_783513,chr1,773513,783513,0,0,0,0,0,0,0,0,1,0,0,1,0,0
chr1_812734_822734,chr1,812734,822734,0,0,0,0,1,0,0,0,0,0,0,0,0,0
chr1_822078_832078,chr1,822078,832078,0,0,0,1,1,0,0,1,2,0,0,0,0,0
chr1_884727_894727,chr1,884727,894727,1,0,0,0,0,0,0,0,0,0,0,1,0,0
chr1_918817_928817,chr1,918817,928817,0,0,0,1,0,0,0,1,1,0,0,3,0,0
chr1_936867_946867,chr1,936867,946867,0,0,0,2,2,0,0,1,1,0,0,0,0,0


In [10]:
C2binfreq2 <- Allbinfreq[,C2names]
C3binfreq2 <- Allbinfreq[,C3names]

In [11]:
head(C2binfreq2)

,U3008,U3031,U3073
,<int>,<int>,<int>
chr1_773513_783513,0,0,1
chr1_812734_822734,0,1,0
chr1_822078_832078,0,1,2
chr1_884727_894727,1,0,0
chr1_918817_928817,0,0,1
chr1_936867_946867,0,2,1


In [12]:
# Reusable function to get intersected top 10% rownames across columns
get_common_top_regions <- function(df) {
  get_top_by_rank <- function(col) {
    # Get the indices of non-zero values
    non_zero_idx <- which(col != 0)
    non_zero_col <- col[non_zero_idx]
    
    # Calculate how many top values we want to select (10% of non-zero values)
    n_top <- max(1, ceiling(length(non_zero_col) * 0.05))
    
    # Get the indices of the top n non-zero values based on the original column
    top_idx <- non_zero_idx[order(non_zero_col, decreasing = TRUE)[1:n_top]]
    
    # Return the rownames based on the top indices
    rownames(df)[top_idx]
  }
  top_lists <- lapply(df, get_top_by_rank)
  Reduce(union, top_lists) # intersect
}
# Apply to your three datasets
hubs_mGC2_filtererd_index <- get_common_top_regions(C2binfreq2)
hubs_mGC3_filtererd_index <- get_common_top_regions(C3binfreq2)

In [13]:
hubs_mGCall_filtererd_index <- get_common_top_regions(cbind(C2binfreq2,C3binfreq2))

In [27]:
hubs_mGCall_filtererd_index %>% length()

[1] 1952

In [28]:
hubs_mGC2_filtererd_index %>% length()

[1] 1443

In [29]:
hubs_mGC3_filtererd_index %>% length()

[1] 1043

In [30]:
unique(c(hubs_mGC2_filtererd_index ,hubs_mGC3_filtererd_index)) %>% length()

[1] 1952

In [31]:
intersect(hubs_mGC2_filtererd_index,hubs_mGC3_filtererd_index)  %>% length()

[1] 534

## barplot

In [32]:
for(i in 1:ncol(C2binfreq2)){
   samplename = colnames(C2binfreq2)[i]
   a <- C2binfreq2[,i] 
   a2 <- as.data.frame(a[a>0])
   colnames(a2) <- 'Connect'
   quantile_5 <- quantile(a2$Connect,probs=0.90)
   numberofhubs <- sum(a2$Connect>quantile_5)
   p1 <- ggplot(a2, aes(x = Connect)) + 
          geom_histogram(binwidth = 1,fill='red') +
          geom_vline(xintercept = quantile_5, 
                     color = "black", 
                     linetype = "dashed", 
                     linewidth = 1) + 
          annotate('text',x=quantile_5,y=2000,label=paste0('Top 5% Hubs = ',numberofhubs),hjust= -1)+
            theme_bw()+ylab('Frequency')+ggtitle(samplename)+theme(plot.title = element_text(hjust=0.5))
    ggsave(plot = p1, filename = paste0(samplename,".barplot.pdf"), width = 4, height = 4)
    
   }

In [33]:
for(i in 1:ncol(C3binfreq2)){
   samplename = colnames(C3binfreq2)[i]
   a <- C3binfreq2[,i]
   a2 <- as.data.frame(a[a>0])
   colnames(a2) <- 'Connect'
   quantile_5 <- quantile(a2$Connect,probs=0.90)
   numberofhubs <- sum(a2$Connect>quantile_5)
   p1 <- ggplot(a2, aes(x = Connect)) + 
          geom_histogram(binwidth = 1,fill='red') +
          geom_vline(xintercept = quantile_5, 
                     color = "black", 
                     linetype = "dashed", 
                     linewidth = 1) + 
          annotate('text',x=quantile_5,y=2000,label=paste0('Top 5% Hubs = ',numberofhubs),hjust= -1)+
            theme_bw()+ylab('Frequency')+ggtitle(samplename)+theme(plot.title = element_text(hjust=0.5))
    ggsave(plot = p1, filename = paste0(samplename,".barplot.pdf"), width = 4, height = 4)
   }

In [34]:
C2binfreq2[hubs_mGC2_filtererd_index,,drop=F] -> hubs_mGC2_filtererdtop10
C3binfreq2[hubs_mGC3_filtererd_index,,drop=F] -> hubs_mGC3_filtererdtop10

In [35]:
length(hubs_mGC2_filtererd_index);
length(hubs_mGC3_filtererd_index);

[1] 1443

[1] 1043

In [36]:
hubs_mGC2_filtererdtop10  %>% rownames() -> mGC2_rownames
hubs_mGC3_filtererdtop10  %>% rownames() -> mGC3_rownames

In [37]:
library(VennDiagram)
library("ggvenn")

Loading required package: grid

Loading required package: futile.logger



In [39]:
pdf("Venn.Specific.Peaks.pdf", width=4, height=4)
options(repr.plot.height=6,repr.plot.width=6)
venn.plot <- venn.diagram(
  x = list(
    mGC2 = mGC2_rownames,
    mGC3 = mGC3_rownames
  ),
  filename = NULL,  # Important: do NOT write directly to file here
  fill = NULL,
  alpha = 0.5,
  cex = 1.5,
  cat.cex = 1.5,
  #cat.pos = c(-20, 20, 0)
)
grid.draw(venn.plot)
dev.off()

pdf 
  2

In [40]:
mGC2_rownames  %>% setdiff(mGC3_rownames)  %>% hubs_mGC2_filtererdtop10[.,,drop=F] -> hubs_mGC2_filtererdtop10
mGC3_rownames  %>% setdiff(mGC2_rownames) %>% hubs_mGC3_filtererdtop10[.,,drop=F] -> hubs_mGC3_filtererdtop10

In [41]:
head(hubs_mGC2_filtererdtop10)

,U3008,U3031,U3073
,<int>,<int>,<int>
chr7_154203330_154213330,50,0,0
chr7_57288576_57298576,45,0,0
chr7_56467639_56477639,43,0,0
chr7_64030095_64040095,34,0,0
chr7_64338278_64348278,33,0,0
chr7_154876426_154886426,33,0,0


In [42]:
mGC2_rownames  %>% intersect(mGC3_rownames) -> shared_names

In [43]:
c(rownames(hubs_mGC2_filtererdtop10),rownames(hubs_mGC3_filtererdtop10)) %>% unique() -> allnames

In [44]:
mGC2_rownames %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame()  %>% 
    write.table(file = "GC2.all.hub.bed",row.names = F,sep='\t',quote = F,col.names=F)
mGC3_rownames %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame()  %>% 
    write.table(file = "GC3.all.hub.bed",row.names = F,sep='\t',quote = F,col.names=F)

In [45]:
hubs_mGC2_filtererdtop10  %>% rownames() %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame() %>%
           write.table(file = "GC2.spec.hub.bed",row.names = F,sep='\t',quote = F,col.names=F)
hubs_mGC3_filtererdtop10  %>% rownames() %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame() %>% 
           write.table(file = "GC3.spec.hub.bed",row.names = F,sep='\t',quote = F,col.names=F)

In [46]:
hubs_mGCall_filtererd_index %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame() %>%
           write.table(file = "All.hubs.bed",row.names = F,sep='\t',quote = F,col.names=F)

In [844]:
saveRDS(hubs_mGC2_filtererdtop10,file = "hubs_mGC2_specHubs.Rds")
saveRDS(hubs_mGC3_filtererdtop10,file = "hubs_mGC2_specHubs.Rds")

In [845]:
saveRDS(mGC2_rownames,file = "hubs_mGC2_Hubs.Rds")
saveRDS(mGC3_rownames,file = "hubs_mGC2_Hubs.Rds")

In [846]:
head(shared_names)

[1] "chr7_100866323_100876323" "chr7_92826642_92836642"  
[3] "chr16_2149929_2159929"    "chr16_29809934_29819934" 
[5] "chr7_100427395_100437395" "chr12_55718971_55728971"

In [847]:
shared_names  %>% str_split('_')  %>% do.call(rbind,.)  %>% as.data.frame() %>%
           write.table(file = "Shared_hubs.bed",row.names = F,sep='\t',quote = F,col.names=F)